In [1]:
import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
REPLAY_JSON_PATH = PROJECT_ROOT / "replay_parser" / "output" / "parsed_match.json"

with open(REPLAY_JSON_PATH, "r") as f:
    replay_data = json.load(f)

print("Top-level keys:", list(replay_data.keys()))
for key, value in replay_data.items():
    print(f"{key}: {len(value)} entries")

Top-level keys: ['players', 'deaths', 'structures_destroyed', 'patron_events', 'mid_boss_kills', 'rift_captures', 'urn_pickups', 'rejuvenator_claims', 'stat_snapshots']
players: 12 entries
deaths: 120 entries
structures_destroyed: 19 entries
patron_events: 1 entries
mid_boss_kills: 3 entries
rift_captures: 6 entries
urn_pickups: 10 entries
rejuvenator_claims: 18 entries
stat_snapshots: 205 entries


In [2]:
# team for players
players_df = pd.DataFrame(replay_data["players"])
controller_to_team = dict(zip(players_df["controller_index"], players_df["team"]))
print(players_df)

# long format DataFrame
rows = []
for snapshot in replay_data["stat_snapshots"]:
    t = snapshot["game_time_s"]
    for p in snapshot["players"]:
        rows.append({
            "game_time_s": t,
            "controller_index": p["controller_index"],
            "team": p["team"],
            "kills": p["kills"],
            "deaths": p["deaths"],
            "assists": p["assists"],
            "hero_damage": p["hero_damage"],
            "objective_damage": p["objective_damage"],
            "gold_net_worth": p["gold_net_worth"],
        })

snapshots_long = pd.DataFrame(rows)
print(snapshots_long.shape)
snapshots_long.head(10)

    controller_index  pawn_index  team            username
0                  2        3429     2             seiryuu
1                  3        3290     2             Numlegs
2                  4        3568     2  Demui on Twitch :D
3                  5        3495     2                Tech
4                  6          77     3               Docha
5                  7          84     3              Specty
6                  8        3475     2            Seikatsu
7                  9          97     3                Anri
8                 10          98     3               Robin
9                 11          94     3     Little Boy 1945
10                12          89     2              Golova
11                13        3534     3          privativev
(2435, 9)


,game_time_s,controller_index,team,kills,deaths,assists,hero_damage,objective_damage,gold_net_worth
0,15.03125,5,2,0,0,0,0,0,600
1,15.03125,6,3,0,0,0,0,0,600
2,15.03125,10,3,0,0,0,0,0,600
3,15.03125,11,3,0,0,0,0,0,600
4,15.03125,13,3,0,0,0,0,0,600
5,30.03125,5,2,0,0,0,0,0,600
6,30.03125,6,3,0,0,0,0,0,600
7,30.03125,10,3,0,0,0,0,0,600
8,30.03125,11,3,0,0,0,0,0,600
9,30.03125,12,2,0,0,0,0,0,600


In [3]:
team_snapshots = (
    snapshots_long
    .groupby(["game_time_s", "team"], as_index=False)
    .agg({
        "kills": "sum",
        "deaths": "sum",
        "assists": "sum",
        "hero_damage": "sum",
        "objective_damage": "sum",
        "gold_net_worth": "sum",
    })
)

print(team_snapshots.shape)
print(team_snapshots.head(20))

(408, 8)
    game_time_s  team  kills  deaths  assists  hero_damage  objective_damage  \
0      15.03125     2      0       0        0            0                 0   
1      15.03125     3      0       0        0            0                 0   
2      30.03125     2      0       0        0            0                 0   
3      30.03125     3      0       0        0            0                 0   
4      45.03125     2      0       0        0            0                 0   
5      45.03125     3      0       0        0            0                 0   
6      60.03125     2      0       0        0          213                 0   
7      60.03125     3      0       0        0          300                 0   
8      75.03125     2      0       0        0          887                 0   
9      75.03125     3      0       0        0          640                 0   
10     90.03125     2      0       0        0         1071                 0   
11     90.03125     3      0   

In [4]:
print(team_snapshots.tail(10))

     game_time_s  team  kills  deaths  assists  hero_damage  objective_damage  \
398  3000.046875     2     55      55      110       292164             21858   
399  3000.046875     3     55      55      106       264389             60243   
400  3015.046875     2     55      55      110       296633             21858   
401  3015.046875     3     55      55      106       267175             64569   
402  3030.046875     2     56      57      111       303760             21858   
403  3030.046875     3     57      56      116       277521             67049   
404  3045.046875     2     56      57      111       303760             21858   
405  3045.046875     3     57      56      116       277521             67049   
406  3060.046875     2     56      57      111       303760             21858   
407  3060.046875     3     57      56      116       277521             67049   

     gold_net_worth  
398          315532  
399          376216  
400          315661  
401          389154 

In [5]:
team_wide = (
    team_snapshots
    .pivot(index="game_time_s", columns="team")
)

team_wide.columns = [
    f"{stat}_team{team}"
    for stat, team in team_wide.columns
]

team_wide = team_wide.reset_index()

print(team_wide.shape)
print(team_wide.head(10))

(204, 13)
   game_time_s  kills_team2  kills_team3  deaths_team2  deaths_team3  \
0     15.03125            0            0             0             0   
1     30.03125            0            0             0             0   
2     45.03125            0            0             0             0   
3     60.03125            0            0             0             0   
4     75.03125            0            0             0             0   
5     90.03125            0            0             0             0   
6    105.03125            0            0             0             0   
7    120.03125            0            0             0             0   
8    135.03125            1            0             0             1   
9    150.03125            1            0             0             1   

   assists_team2  assists_team3  hero_damage_team2  hero_damage_team3  \
0              0              0                  0                  0   
1              0              0                  0 

In [6]:
diff_df = pd.DataFrame({
    "game_time_s": team_wide["game_time_s"],

    "kills_diff": (
        team_wide["kills_team2"] -
        team_wide["kills_team3"]
    ),

    "deaths_diff": (
        team_wide["deaths_team2"] -
        team_wide["deaths_team3"]
    ),

    "assists_diff": (
        team_wide["assists_team2"] -
        team_wide["assists_team3"]
    ),

    "hero_damage_diff": (
        team_wide["hero_damage_team2"] -
        team_wide["hero_damage_team3"]
    ),

    "objective_damage_diff": (
        team_wide["objective_damage_team2"] -
        team_wide["objective_damage_team3"]
    ),

    "net_worth_diff": (
        team_wide["gold_net_worth_team2"] -
        team_wide["gold_net_worth_team3"]
    ),
})

print(diff_df.shape)
print(diff_df.head(10))

(204, 7)
   game_time_s  kills_diff  deaths_diff  assists_diff  hero_damage_diff  \
0     15.03125           0            0             0                 0   
1     30.03125           0            0             0                 0   
2     45.03125           0            0             0                 0   
3     60.03125           0            0             0               -87   
4     75.03125           0            0             0               247   
5     90.03125           0            0             0               378   
6    105.03125           0            0             0               402   
7    120.03125           0            0             0              -326   
8    135.03125           1           -1             1               154   
9    150.03125           1           -1             1              -381   

   objective_damage_diff  net_worth_diff  
0                      0           -1800  
1                      0           -1200  
2                      0            

In [7]:
# replay event lists into DataFrames
deaths_df = pd.DataFrame(replay_data["deaths"])
structures_df = pd.DataFrame(replay_data["structures_destroyed"])
mid_boss_df = pd.DataFrame(replay_data["mid_boss_kills"])
rejuvenator_df = pd.DataFrame(replay_data["rejuvenator_claims"])

print("Deaths:")
print(deaths_df.head())

print("\nStructures:")
print(structures_df.head())

print("\nMid boss:")
print(mid_boss_df)

print("\nRejuvenator claims:")
print(rejuvenator_df.head(20))

Deaths:
   team  player_index            username  game_time_s  death_duration_s  \
0     3            94     Little Boy 1945   128.187500          8.062500   
1     2            89              Golova   151.484375          8.062500   
2     3            94     Little Boy 1945   214.687500          8.046875   
3     2            92  Demui on Twitch :D   323.109375          8.062500   
4     3            94     Little Boy 1945   324.718750          8.062500   

  respawn_full_health  
0               False  
1               False  
2               False  
3               False  
4               False  

Structures:
   structure_type  team  destroyed_time_s
0  Tier2Structure     2        670.609375
1  Tier2Structure     3        784.265625
2  Tier2Structure     3        869.343750
3  Tier1Structure     2        903.500000
4  Tier2Structure     2        934.390625

Mid boss:
   killed_time_s
0    1598.609375
1    2563.921875
2    2954.109375

Rejuvenator claims:
    controller_index  team

In [8]:
team_mapping = {
    2: "Team0",
    3: "Team1"
}

mid_boss_with_team = []

for _, boss in mid_boss_df.iterrows():
    boss_time = boss["killed_time_s"]

    claims_after_boss = rejuvenator_df[
        (rejuvenator_df["game_time_s"] >= boss_time) &
        (rejuvenator_df["game_time_s"] <= boss_time + 30)
    ].sort_values("game_time_s")

    team_claimed = claims_after_boss.iloc[0]["team"] if len(claims_after_boss) > 0 else None

    mid_boss_with_team.append({
        "killed_time_s": boss_time,
        "team_claimed": team_claimed,
        "perspective_team": team_mapping.get(team_claimed) if team_claimed else None,
    })

mid_boss_df = pd.DataFrame(mid_boss_with_team)
print(mid_boss_df)

   killed_time_s  team_claimed perspective_team
0    1598.609375           2.0            Team0
1    2563.921875           3.0            Team1
2    2954.109375           3.0            Team1


In [9]:
FAST_RESPAWN_THRESHOLD_S = 4.0 # some extra time just in case
REJUV_WINDOW_S = 180  # 3 mins

def team_had_active_rejuv(row, rejuv_df, window_s=REJUV_WINDOW_S):
    team_claims = rejuv_df[rejuv_df["team"] == row["team"]]
    active = team_claims[
        (team_claims["game_time_s"] <= row["game_time_s"]) &
        (row["game_time_s"] <= team_claims["game_time_s"] + window_s)
    ]
    return not active.empty

deaths_df["likely_rejuv_used"] = deaths_df["death_duration_s"] < FAST_RESPAWN_THRESHOLD_S

deaths_df["rejuv_buff_active"] = deaths_df.apply(
    lambda r: team_had_active_rejuv(r, rejuvenator_df), axis=1
)

deaths_df["credit_used"] = deaths_df["likely_rejuv_used"] & deaths_df["rejuv_buff_active"]

rejuv_usage_events = deaths_df[deaths_df["credit_used"]][
    ["team", "player_index", "game_time_s", "death_duration_s"]
]
print(rejuv_usage_events)

     team  player_index  game_time_s  death_duration_s
66      2            78  1705.156250          2.984375
68      2            82  1743.828125          3.000000
71      2            92  1761.109375          2.984375
102     3            94  2729.343750          3.000000
103     3            77  2733.015625          2.984375
113     3           100  2967.468750          3.000000


In [10]:
def assign_credits_to_claims(deaths_df, rejuvenator_df):
    fast_deaths = deaths_df[deaths_df["death_duration_s"] < FAST_RESPAWN_THRESHOLD_S].copy()
    claim_counts = {}

    assignments = []
    for _, d in fast_deaths.iterrows():
        team_claims = rejuvenator_df[
            (rejuvenator_df["team"] == d["team"]) &
            (rejuvenator_df["game_time_s"] <= d["game_time_s"])
        ].sort_values("game_time_s")
        if team_claims.empty:
            continue
        nearest_claim_time = team_claims.iloc[-1]["game_time_s"]
        if d["game_time_s"] - nearest_claim_time > REJUV_WINDOW_S:
            continue
        used_so_far = claim_counts.get(nearest_claim_time, 0)
        if used_so_far >= 3:
            continue
        claim_counts[nearest_claim_time] = used_so_far + 1
        assignments.append(d["game_time_s"])

    return claim_counts

credit_counts_by_claim_time = assign_credits_to_claims(deaths_df, rejuvenator_df)
print(credit_counts_by_claim_time)

{np.float64(1605.296875): 3, np.float64(2570.359375): 2, np.float64(2960.78125): 1}


In [11]:
team_wide = team_wide.rename(columns={
    col: col.replace("_team2", "_team0").replace("_team3", "_team1")
    for col in team_wide.columns
})
print(team_wide.columns.tolist())

['game_time_s', 'kills_team0', 'kills_team1', 'deaths_team0', 'deaths_team1', 'assists_team0', 'assists_team1', 'hero_damage_team0', 'hero_damage_team1', 'objective_damage_team0', 'objective_damage_team1', 'gold_net_worth_team0', 'gold_net_worth_team1']


In [12]:
structures_df["destroying_team"] = structures_df["team"].map({2: 3, 3: 2})  # whoever didn't own it destroyed it
structures_df["perspective_team"] = structures_df["destroying_team"].map(team_mapping)

# cumulative count over time per team
structures_df = structures_df.sort_values("destroyed_time_s")
structures_df["team0_cumulative"] = (structures_df["perspective_team"] == "Team0").cumsum()
structures_df["team1_cumulative"] = (structures_df["perspective_team"] == "Team1").cumsum()
print(structures_df[["destroyed_time_s", "team", "destroying_team", "team0_cumulative", "team1_cumulative"]])

    destroyed_time_s  team  destroying_team  team0_cumulative  \
0         670.609375     2                3                 0   
1         784.265625     3                2                 1   
2         869.343750     3                2                 2   
3         903.500000     2                3                 2   
4         934.390625     2                3                 2   
5         980.875000     2                3                 2   
6        1129.593750     3                2                 3   
7        1191.781250     3                2                 4   
8        1633.406250     2                3                 4   
9        1672.765625     3                2                 5   
10       1778.203125     2                3                 5   
11       2174.671875     2                3                 5   
12       2347.703125     2                3                 5   
13       2567.593750     2                3                 5   
14       2572.781250     

In [13]:
mid_boss_df = mid_boss_df.sort_values("killed_time_s")
mid_boss_df["team0_cumulative"] = (mid_boss_df["perspective_team"] == "Team0").cumsum()
mid_boss_df["team1_cumulative"] = (mid_boss_df["perspective_team"] == "Team1").cumsum()
print(mid_boss_df)

   killed_time_s  team_claimed perspective_team  team0_cumulative  \
0    1598.609375           2.0            Team0                 1   
1    2563.921875           3.0            Team1                 1   
2    2954.109375           3.0            Team1                 1   

   team1_cumulative  
0                 0  
1                 1  
2                 2  


In [14]:
team_wide = team_wide.sort_values("game_time_s")
structures_df = structures_df.sort_values("destroyed_time_s")
mid_boss_df = mid_boss_df.sort_values("killed_time_s")

merged = pd.merge_asof(
    team_wide,
    structures_df[["destroyed_time_s", "team0_cumulative", "team1_cumulative"]].rename(
        columns={"team0_cumulative": "own_objectives_cumulative", "team1_cumulative": "enemy_objectives_cumulative"}
    ),
    left_on="game_time_s",
    right_on="destroyed_time_s",
    direction="backward",
)

merged = pd.merge_asof(
    merged,
    mid_boss_df[["killed_time_s", "team0_cumulative", "team1_cumulative"]].rename(
        columns={"team0_cumulative": "own_mid_boss_claims", "team1_cumulative": "enemy_mid_boss_claims"}
    ),
    left_on="game_time_s",
    right_on="killed_time_s",
    direction="backward",
)

# fill NaNs with 0
for col in ["own_objectives_cumulative", "enemy_objectives_cumulative", "own_mid_boss_claims", "enemy_mid_boss_claims"]:
    merged[col] = merged[col].fillna(0).astype("int64")

merged = merged.drop(columns=["destroyed_time_s", "killed_time_s"])
print(merged.shape)
print(merged.head(10))

(204, 17)
   game_time_s  kills_team0  kills_team1  deaths_team0  deaths_team1  \
0     15.03125            0            0             0             0   
1     30.03125            0            0             0             0   
2     45.03125            0            0             0             0   
3     60.03125            0            0             0             0   
4     75.03125            0            0             0             0   
5     90.03125            0            0             0             0   
6    105.03125            0            0             0             0   
7    120.03125            0            0             0             0   
8    135.03125            1            0             0             1   
9    150.03125            1            0             0             1   

   assists_team0  assists_team1  hero_damage_team0  hero_damage_team1  \
0              0              0                  0                  0   
1              0              0                  0 

In [15]:
import torch
import torch.nn as nn

# load the model
class WinProbabilityLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=32, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        logits = self.fc(lstm_out)
        return logits.squeeze(-1)

MODEL_PATH = PROJECT_ROOT / "models" / "win_probability_lstm_v1.pt"
checkpoint = torch.load(MODEL_PATH, map_location="cpu")

feature_cols = checkpoint["feature_cols"]

model = WinProbabilityLSTM(
    input_size=checkpoint["input_size"],
    hidden_size=checkpoint["hidden_size"],
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

def get_user_team(username, players_df):
    match = players_df[players_df["username"] == username]
    if match.empty:
        raise ValueError(f"No player found with username '{username}'")
    return match.iloc[0]["team"]  # Returns 2 or 3

USER_USERNAME = "seiryuu" 
USER_RAW_TEAM = get_user_team(USER_USERNAME, players_df)
print(f"Viewing report as {USER_USERNAME}, raw team {USER_RAW_TEAM}\n")

own_col = "team0" if USER_RAW_TEAM == 2 else "team1"
enemy_col = "team1" if USER_RAW_TEAM == 2 else "team0"

# Dynamically calculate diffs based on the user's perspective
merged['net_worth_diff'] = merged[f'gold_net_worth_{own_col}'] - merged[f'gold_net_worth_{enemy_col}']
merged['kills_diff'] = merged[f'kills_{own_col}'] - merged[f'kills_{enemy_col}']
merged['deaths_diff'] = merged[f'deaths_{own_col}'] - merged[f'deaths_{enemy_col}']
merged['damage_diff'] = merged[f'hero_damage_{own_col}'] - merged[f'hero_damage_{enemy_col}']

# Swap cumulative objectives if user is on Team 1 (since previous cells hardcoded them for Team 0)
if USER_RAW_TEAM == 3:
    merged[['own_objectives_cumulative', 'enemy_objectives_cumulative']] = merged[['enemy_objectives_cumulative', 'own_objectives_cumulative']].values
    merged[['own_mid_boss_claims', 'enemy_mid_boss_claims']] = merged[['enemy_mid_boss_claims', 'own_mid_boss_claims']].values

merged['objectives_diff'] = merged['own_objectives_cumulative'] - merged['enemy_objectives_cumulative']
merged['mid_boss_diff'] = merged['own_mid_boss_claims'] - merged['enemy_mid_boss_claims']

merged['own_deaths_cumulative'] = merged[f'deaths_{own_col}']
merged['enemy_deaths_cumulative'] = merged[f'deaths_{enemy_col}']
merged['deaths_cumulative_diff'] = merged['own_deaths_cumulative'] - merged['enemy_deaths_cumulative']

feature_array = merged[feature_cols].values.astype("float32")
input_tensor = torch.tensor(feature_array).unsqueeze(0)  # [1, seq_len, input_size]

with torch.no_grad():
    logits = model(input_tensor)
    win_probs = torch.sigmoid(logits).squeeze(0).numpy()

# Save directly as your own win probability
merged["win_prob_own"] = win_probs
print(merged[["game_time_s", "win_prob_own"]].head())

Viewing report as seiryuu, raw team 2

   game_time_s  win_prob_own
0     15.03125      0.188125
1     30.03125      0.177117
2     45.03125      0.352302
3     60.03125      0.416144
4     75.03125      0.401653


In [16]:
def get_dynamic_threshold(game_time_s):
    minutes = game_time_s / 60
    if minutes < 10:
        return 0.10   # First 10 mins: Require a massive 10% swing to care
    elif minutes < 25:
        return 0.07   # 10 - 25 mins: Require a 7% swing
    else:
        return 0.04   # 25+ mins: Highly sensitive, 4% swings are critical

# Apply smoothing to quiet the early-game noise
merged["win_prob_smoothed"] = merged["win_prob_own"].rolling(window=3, center=True, min_periods=1).mean()
merged["win_prob_delta_smoothed"] = merged["win_prob_smoothed"].diff()

# Map the dynamic threshold to each row based on its timestamp
dynamic_thresholds = merged["game_time_s"].apply(get_dynamic_threshold)

# Filter out the swings that don't meet the time-specific threshold
swings_smoothed = merged[merged["win_prob_delta_smoothed"].abs() >= dynamic_thresholds][
    ["game_time_s", "win_prob_smoothed", "win_prob_delta_smoothed"]
]

print(f"Found {len(swings_smoothed)} meaningful narrative swings using dynamic scaling:")
print(swings_smoothed.head(10))

Found 17 meaningful narrative swings using dynamic scaling:
     game_time_s  win_prob_smoothed  win_prob_delta_smoothed
8     135.031250           0.476778                 0.112270
11    180.031250           0.296822                -0.168586
13    210.031250           0.306997                 0.108101
14    225.031250           0.471146                 0.164149
15    240.046875           0.608233                 0.137086
19    300.046875           0.679055                 0.131015
43    660.046875           0.586779                -0.097846
46    705.046875           0.729761                 0.097738
132  1995.046875           0.629361                -0.152292
133  2010.046875           0.501256                -0.128104


In [17]:
urn_df = pd.DataFrame(replay_data["urn_pickups"])
rift_df = pd.DataFrame(replay_data["rift_captures"])

print(urn_df.head())
print(rift_df.head())

   spawn_time_s  pickup_time_s
0     630.03125     681.796875
1     930.03125     942.671875
2    1230.03125    1251.968750
3    1530.03125    1766.250000
4    1830.03125    1842.828125
   cashin_time_s  spawn_time_s
0     744.093750    724.000000
1    1148.296875   1128.203125
2    1659.015625   1638.921875
3    1984.687500   1964.593750
4    2520.796875   2500.703125


In [18]:
def infer_team_from_net_worth_spike(event_time, window_s=5):
    before = merged[merged["game_time_s"] <= event_time].tail(1)
    after = merged[merged["game_time_s"] >= event_time].head(1)

    if before.empty or after.empty:
        return None

    # raw gold values
    delta_0 = after["gold_net_worth_team0"].values[0] - before["gold_net_worth_team0"].values[0]
    delta_1 = after["gold_net_worth_team1"].values[0] - before["gold_net_worth_team1"].values[0]

    if delta_0 > delta_1:
        return 2  # Team 2
    elif delta_1 > delta_0:
        return 3  # Team 3
    return None

rift_df["team"] = rift_df["cashin_time_s"].apply(infer_team_from_net_worth_spike)
urn_df["team"] = urn_df["pickup_time_s"].apply(infer_team_from_net_worth_spike)

In [19]:
pawn_to_username = {
    p["pawn_index"]: p["username"]
    for p in replay_data["players"]
    if p.get("pawn_index") is not None
}
print(pawn_to_username)

{3429: 'seiryuu', 3290: 'Numlegs', 3568: 'Demui on Twitch :D', 3495: 'Tech', 77: 'Docha', 84: 'Specty', 3475: 'Seikatsu', 97: 'Anri', 98: 'Robin', 94: 'Little Boy 1945', 89: 'Golova', 3534: 'privativev'}


In [20]:
print(deaths_df[["player_index", "username"]].drop_duplicates())
print("\nDeaths with missing username:", deaths_df["username"].isna().sum())

    player_index            username
0             94     Little Boy 1945
1             89              Golova
3             92  Demui on Twitch :D
5             97                Anri
6             82             seiryuu
9             85            Seikatsu
13            77               Docha
17           100          privativev
24            84              Specty
31            98               Robin
46            80                Tech
50            78             Numlegs

Deaths with missing username: 0


In [21]:
def find_nearest_events(swing_time, lookback_s=30):
    events = []

    nearby_deaths = deaths_df[
        (deaths_df["game_time_s"] <= swing_time) &
        (deaths_df["game_time_s"] >= swing_time - lookback_s)
    ]
    for _, d in nearby_deaths.iterrows():
        events.append({
            "type": "death",
            "time": d["game_time_s"],
            "team": d["team"],
            "player_index": d["player_index"],
            "username": d["username"],
        })

    nearby_structures = structures_df[
        (structures_df["destroyed_time_s"] <= swing_time) &
        (structures_df["destroyed_time_s"] >= swing_time - lookback_s)
    ]
    for _, s in nearby_structures.iterrows():
        events.append({"type": "structure", "time": s["destroyed_time_s"], "team": s["team"], "structure_type": s["structure_type"]})

    nearby_midboss = mid_boss_df[
        (mid_boss_df["killed_time_s"] <= swing_time) &
        (mid_boss_df["killed_time_s"] >= swing_time - lookback_s)
    ]
    for _, mb in nearby_midboss.iterrows():
        events.append({"type": "mid_boss", "time": mb["killed_time_s"], "team_claimed": mb["team_claimed"]})

    nearby_rejuv = rejuvenator_df[
        (rejuvenator_df["game_time_s"] <= swing_time) &
        (rejuvenator_df["game_time_s"] >= swing_time - lookback_s)
    ]
    for _, r in nearby_rejuv.iterrows():
        events.append({"type": "rejuv_claim", "time": r["game_time_s"], "team": r["team"]})

    nearby_urns = urn_df[
        (urn_df["pickup_time_s"] <= swing_time) &
        (urn_df["pickup_time_s"] >= swing_time - lookback_s)
    ]
    for _, u in nearby_urns.iterrows():
        events.append({"type": "urn_pickup", "time": u["pickup_time_s"], "team": u["team"]})

    nearby_rifts = rift_df[
        (rift_df["cashin_time_s"] <= swing_time) &
        (rift_df["cashin_time_s"] >= swing_time - lookback_s)
    ]
    for _, r in nearby_rifts.iterrows():
        events.append({"type": "rift_capture", "time": r["cashin_time_s"], "team": r["team"]})

    return sorted(events, key=lambda e: e["time"], reverse=True)


attributed_swings = []

for _, row in swings_smoothed.iterrows():
    swing_time = row["game_time_s"]
    delta = row["win_prob_delta_smoothed"]
    events = find_nearest_events(swing_time)

    attributed_swings.append({
        "swing_time_s": swing_time,
        "win_prob_after": row["win_prob_smoothed"],
        "delta": delta,
        "direction": "team0_favored" if delta > 0 else "team1_favored",
        "nearest_events": events,
    })

for s in attributed_swings:
    print(f"\n--- Swing at {s['swing_time_s']:.1f}s ({s['direction']}, Δ={s['delta']:+.3f}) ---")
    if not s["nearest_events"]:
        print("  No nearby events found within lookback window")
    for e in s["nearest_events"]:
        print(f"  {e}")


--- Swing at 135.0s (team0_favored, Δ=+0.112) ---
  {'type': 'death', 'time': 128.1875, 'team': 3, 'player_index': 94, 'username': 'Little Boy 1945'}

--- Swing at 180.0s (team1_favored, Δ=-0.169) ---
  {'type': 'death', 'time': 151.484375, 'team': 2, 'player_index': 89, 'username': 'Golova'}

--- Swing at 210.0s (team0_favored, Δ=+0.108) ---
  No nearby events found within lookback window

--- Swing at 225.0s (team0_favored, Δ=+0.164) ---
  {'type': 'death', 'time': 214.6875, 'team': 3, 'player_index': 94, 'username': 'Little Boy 1945'}

--- Swing at 240.0s (team0_favored, Δ=+0.137) ---
  {'type': 'death', 'time': 214.6875, 'team': 3, 'player_index': 94, 'username': 'Little Boy 1945'}

--- Swing at 300.0s (team0_favored, Δ=+0.131) ---
  No nearby events found within lookback window

--- Swing at 660.0s (team1_favored, Δ=-0.098) ---
  {'type': 'death', 'time': 647.625, 'team': 3, 'player_index': 94, 'username': 'Little Boy 1945'}
  {'type': 'death', 'time': 644.765625, 'team': 2, 'pla

In [22]:
def find_followup_structure_after_death(death_team, death_time, structures_df, window_s=90):
    candidates = structures_df[
        (structures_df["team"] == death_team) &
        (structures_df["destroyed_time_s"] >= death_time) &
        (structures_df["destroyed_time_s"] <= death_time + window_s)
    ].sort_values("destroyed_time_s")
    return candidates.iloc[0] if not candidates.empty else None

In [23]:
import pandas as pd

STRUCTURE_DISPLAY_NAMES = {
    "Tier1Structure": "Guardian",
    "Tier2Structure": "Walker",
    "BaseGuardian": "Base Guardian",
    "InnerTower": "Shrine",
    "Patron": "Patron"
}

def format_time(seconds):
    m = int(seconds // 60)
    s = int(seconds % 60)
    return f"{m}:{s:02d}"

def get_win_prob_pct(prob):
    return prob * 100 

def structure_name(event_or_row):
    # Handle dictionary (from nearest_events) or Pandas Series (from followup df)
    if isinstance(event_or_row, dict):
        raw_type = event_or_row.get('structure_type', '')
        lane = event_or_row.get('lane')
    else:
        raw_type = event_or_row['structure_type']
        lane = event_or_row.get('lane') if 'lane' in event_or_row else None
        
    base_name = STRUCTURE_DISPLAY_NAMES.get(raw_type, raw_type)
    
    # Apply lane prefix if lane data exists
    if lane is not None and pd.notna(lane):
        lane_color = str(lane).capitalize()
        return f"{lane_color} {base_name}"
        
    return base_name

def get_event_polarity(event, user_raw_team):
    is_user_team = event.get("team") == user_raw_team
    
    if event["type"] == "death":
        return -1 if is_user_team else 1
    elif event["type"] == "structure":
        return -1 if is_user_team else 1 # event["team"] is the loser of the structure
    elif event["type"] == "mid_boss":
        return 1 if event["team_claimed"] == user_raw_team else -1
    elif event["type"] in ["urn_pickup", "rift_capture"]:
        return 1 if event["team"] == user_raw_team else -1
        
    return 0

def event_to_phrase(event, user_raw_team):
    is_user_team = event.get("team") == user_raw_team

    if event["type"] == "death":
        who = event["username"]
        possessive = "your teammate" if is_user_team else "the enemy"
        return f"{possessive} {who} died at {format_time(event['time'])}"

    if event["type"] == "structure":
        if is_user_team:
            return f"the enemy destroyed your {structure_name(event)} at {format_time(event['time'])}"
        else:
            return f"your team destroyed an enemy {structure_name(event)} at {format_time(event['time'])}"

    if event["type"] == "mid_boss":
        claimer = "your team" if event["team_claimed"] == user_raw_team else "the enemy"
        return f"{claimer} claimed the Mid Boss Rejuvenator at {format_time(event['time'])}"

    if event["type"] == "urn_pickup" or event["type"] == "rift_capture":
        claimer = "your team" if event["team"] == user_raw_team else "the enemy"
        action = "secured the Urn" if event["type"] == "urn_pickup" else "cashed in the Rift"
        return f"{claimer} {action} at {format_time(event['time'])}"

    return None

def find_followup_structure(claiming_team, after_time, structures_df, window_s=90):
    candidates = structures_df[
        (structures_df["team"] != claiming_team) &
        (structures_df["destroyed_time_s"] >= after_time) &
        (structures_df["destroyed_time_s"] <= after_time + window_s)
    ].sort_values("destroyed_time_s")
    return candidates.iloc[0] if not candidates.empty else None

def find_followup_structure_after_death(death_team, death_time, structures_df, window_s=90):
    candidates = structures_df[
        (structures_df["team"] == death_team) &
        (structures_df["destroyed_time_s"] >= death_time) &
        (structures_df["destroyed_time_s"] <= death_time + window_s)
    ].sort_values("destroyed_time_s")
    return candidates.iloc[0] if not candidates.empty else None

QUIET_NET_CHANGE_THRESHOLD = 15

def build_full_timeline(attributed_swings, mid_boss_df, deaths_df, user_raw_team):
    timeline = []
    for s in attributed_swings:
        timeline.append({"time": s["swing_time_s"], "kind": "swing", "data": s})
        
    for _, mb in mid_boss_df.iterrows():
        matching_claims = rejuvenator_df[
            (rejuvenator_df["team"] == mb["team_claimed"]) &
            (rejuvenator_df["game_time_s"] >= mb["killed_time_s"])
        ].sort_values("game_time_s")
        claim_time = matching_claims["game_time_s"].iloc[0] if not matching_claims.empty else None
        credits = credit_counts_by_claim_time.get(claim_time, 0) if claim_time is not None else 0
        timeline.append({
            "time": mb["killed_time_s"],
            "kind": "mid_boss",
            "data": {"team_claimed": mb["team_claimed"], "credits_used": credits}
        })
    timeline.sort(key=lambda x: x["time"])
    return timeline

def generate_narrative(attributed_swings, mid_boss_df, deaths_df, user_raw_team, merged_df, structures_df):
    timeline = build_full_timeline(attributed_swings, mid_boss_df, deaths_df, user_raw_team)
    lines = []
    seen_events = set()

    quiet_start_time = merged_df["game_time_s"].min()
    quiet_start_prob = get_win_prob_pct(merged_df["win_prob_smoothed"].iloc[0])
    last_known_prob = quiet_start_prob

    def flush_quiet(end_time, end_prob):
        nonlocal quiet_start_time, quiet_start_prob
        net_change = end_prob - quiet_start_prob
        if abs(net_change) >= QUIET_NET_CHANGE_THRESHOLD:
            verb = "pulled ahead" if net_change > 0 else "fell behind"
            lines.append(
                f"From {format_time(quiet_start_time)} to {format_time(end_time)}, "
                f"your team steadily {verb} with no single decisive event, "
                f"win chance moving from {quiet_start_prob:.0f}% to {end_prob:.0f}%."
            )
        quiet_start_time = end_time
        quiet_start_prob = end_prob

    biggest_swing = None

    for item in timeline:
        t = item["time"]

        if item["kind"] == "mid_boss":
            flush_quiet(t, last_known_prob)
            claimer = "Your team" if item["data"]["team_claimed"] == user_raw_team else "The enemy"
            line = f"{claimer} claimed the Mid Boss Rejuvenator at {format_time(t)}, gaining a team-wide buff."
            credits = item["data"]["credits_used"]
            if credits > 0:
                line += f" Over the next few minutes, they used {credits} quick-respawn charge{'s' if credits != 1 else ''}."
            followup = find_followup_structure(item["data"]["team_claimed"], t, structures_df)
            if followup is not None:
                line += f" Riding that advantage, they pushed and destroyed a {structure_name(followup)} at {format_time(followup['destroyed_time_s'])}."
            lines.append(line)

        elif item["kind"] == "swing":
            s = item["data"]
            delta_pct = abs(s["delta"]) * 100
            prob_pct = get_win_prob_pct(s["win_prob_after"])
            is_positive_swing = s["delta"] > 0

            event_phrase, event_type, triggering_event = None, None, None
            for e in s["nearest_events"]:
                key = (e["type"], e["time"])
                if key not in seen_events:
                    event_phrase = event_to_phrase(e, user_raw_team)
                    event_type = e["type"]
                    triggering_event = e
                    seen_events.add(key)
                    break

            if event_phrase:
                flush_quiet(t, last_known_prob)
                polarity = get_event_polarity(triggering_event, user_raw_team)

                cap_phrase = event_phrase[0].upper() + event_phrase[1:]

                if is_positive_swing and polarity >= 0:
                    line = f"{cap_phrase}, which boosted your win chance by {delta_pct:.0f}% (now {prob_pct:.0f}% win chance)."
                elif not is_positive_swing and polarity <= 0:
                    line = f"{cap_phrase}, which hurt your win chance by {delta_pct:.0f}% (now {prob_pct:.0f}% win chance)."
                elif is_positive_swing and polarity == -1:
                    line = f"Even though {event_phrase}, your team pulled ahead overall, boosting your win chance by {delta_pct:.0f}% (now {prob_pct:.0f}%)."
                elif not is_positive_swing and polarity == 1:
                    line = f"Even though {event_phrase}, the enemy pressed their advantage overall, dropping your win chance by {delta_pct:.0f}% (now {prob_pct:.0f}%)."

                if event_type == "rift_capture" and polarity == 1 and is_positive_swing:
                    claiming_team = triggering_event["team"]
                    followup = find_followup_structure(claiming_team, triggering_event["time"], structures_df)
                    if followup is not None:
                        claimer = "Your team" if claiming_team == user_raw_team else "The enemy"
                        line += f" {claimer} used the advantage to destroy an enemy {structure_name(followup)} at {format_time(followup['destroyed_time_s'])}."
                
                elif event_type == "death" and polarity == -1 and not is_positive_swing:
                    death_team = triggering_event["team"]
                    followup = find_followup_structure_after_death(death_team, triggering_event["time"], structures_df)
                    if followup is not None:
                        loser = "your team" if death_team == user_raw_team else "the enemy"
                        line += f" With a player down, {loser} lost a {structure_name(followup)} shortly after, at {format_time(followup['destroyed_time_s'])}."

                lines.append(line)
                quiet_start_time = t
                quiet_start_prob = prob_pct

            match_midpoint = merged_df["game_time_s"].max() / 2
            if t >= match_midpoint and not is_positive_swing and polarity == -1:
                if biggest_swing is None or delta_pct > biggest_swing["delta_pct"]:
                    biggest_swing = {"time": triggering_event["time"] if triggering_event else t, "delta_pct": delta_pct, "phrase": event_phrase}

            last_known_prob = prob_pct

    flush_quiet(merged_df["game_time_s"].max(), last_known_prob)

    final_prob = get_win_prob_pct(merged_df["win_prob_smoothed"].iloc[-1])
    outcome = "led" if final_prob >= 50 else "trailed"
    lines.append(f"\nBy the end of the game, your team {outcome} with a {final_prob:.0f}% win chance.")

    if biggest_swing:
        turning_phrase = biggest_swing["phrase"] or "a shift in momentum"
        lines.append(
            f"The fatal turning point came when {turning_phrase}, "
            f"a {biggest_swing['delta_pct']:.0f}% swing that let the enemy press their advantage from there."
        )

    return lines

narrative_lines = generate_narrative(attributed_swings, mid_boss_df, deaths_df, USER_RAW_TEAM, merged, structures_df)
for line in narrative_lines:
    print(line)

The enemy Little Boy 1945 died at 2:08, which boosted your win chance by 11% (now 48% win chance).
Your teammate Golova died at 2:31, which hurt your win chance by 17% (now 30% win chance).
The enemy Little Boy 1945 died at 3:34, which boosted your win chance by 16% (now 47% win chance).
From 3:45 to 11:00, your team steadily pulled ahead with no single decisive event, win chance moving from 47% to 68%.
Even though the enemy Little Boy 1945 died at 10:47, the enemy pressed their advantage overall, dropping your win chance by 10% (now 59%).
Your team secured the Urn at 11:21, which boosted your win chance by 10% (now 73% win chance).
Your team claimed the Mid Boss Rejuvenator at 26:38, gaining a team-wide buff. Riding that advantage, they pushed and destroyed a Guardian at 27:52.
The enemy cashed in the Rift at 33:04, which hurt your win chance by 15% (now 63% win chance).
From 33:15 to 37:30, your team steadily fell behind with no single decisive event, win chance moving from 63% to 37